In [1]:
"""
MAP - Charting Student Math Misunderstandings - Inference v6.0 (Lightweight RAG)
優化與新增重點:
1. 內建純 Python 高效倒排索引 BM25，零外部依賴、極速動態檢索歷史相似案例。
2. 動態 One-Shot RAG Prompt：為每筆測試資料精準貼上 1 個最相似的考古題與答案。
3. 承襲 V5.2 核心提速：所有重排分數全面留在 GPU 內原地計算，拒絕巨大 Logits 搬移至 CPU。
4. 嚴格防禦機制：確保 9 小時內絕對跑完且不發生 IndexError/OOM。
"""

import os
import re
import time
import math
import pandas as pd
import numpy as np
import torch
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from tqdm import tqdm

# ==== 路徑 ====     
BASE_MODEL_PATH = "/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1"
ADAPTER_PATH = "/kaggle/input/datasets/alextsai2004/gemma-math-misunderstanding-lora/best_gemma_lora_model"
TEST_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/test.csv"
TRAIN_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/train.csv"
SAMPLE_CSV = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/sample_submission.csv"
OUTPUT_CSV = "/kaggle/working/submission.csv"

# ==== 超參 (動態 RAG 最佳化設定) ====
BATCH_SIZE = 16           # 同時處理多少筆 test row (Phase 1)
SUB_BATCH_SIZE = 32       # Phase 2 攤平後的 forward 上限
NUM_BEAMS = 5             # beam search 寬度
NUM_RETURN = 5            # 每筆拿多少候選
MAX_NEW_TOKENS = 24       # label 不會超過這麼長

# ==== Sample submission 骨架 ====
sample = pd.read_csv(SAMPLE_CSV)
ROW_ID_COL = sample.columns[0]
PRED_COL = sample.columns[1]
print(f"Sample shape: {sample.shape}")

# ==== 1. 純 Python 超輕量高效 BM25 檢索器 ====
def tokenize_text(text):
    if not isinstance(text, str):
        return []
    return re.findall(r'\w+', text.lower())

class LightweightBM25:
    def __init__(self, docs, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.doc_lens = [len(d) for d in docs]
        self.avg_doc_len = sum(self.doc_lens) / len(docs) if docs else 1
        self.N = len(docs)
        
        # 建立倒排索引 term -> list of (doc_id, tf)
        self.index = {}
        df = {}
        for doc_id, doc in enumerate(docs):
            counts = Counter(doc)
            for term, tf in counts.items():
                if term not in self.index:
                    self.index[term] = []
                self.index[term].append((doc_id, tf))
                df[term] = df.get(term, 0) + 1
        
        # 計算 IDF
        self.idf = {}
        for term, f in df.items():
            self.idf[term] = math.log(1 + (self.N - f + 0.5) / (f + 0.5))
            
    def retrieve_top_1(self, query_tokens):
        """極速檢索最相似的 1 筆訓練集索引"""
        scores = {}
        for term in query_tokens:
            if term not in self.index:
                continue
            idf = self.idf[term]
            for doc_id, tf in self.index[term]:
                num = idf * tf * (self.k1 + 1)
                den = tf + self.k1 * (1 - self.b + self.b * self.doc_lens[doc_id] / self.avg_doc_len)
                scores[doc_id] = scores.get(doc_id, 0.0) + (num / den)
        if not scores:
            return 0  # 若完全沒有匹配，預設回傳第一筆
        return max(scores, key=scores.get)

# ==== 訓練集準備與 BM25 索引建立 ====
print("Loading train set & building BM25 index...")
train_df = pd.read_csv(TRAIN_CSV)
for col in ["QuestionText", "MC_Answer", "StudentExplanation"]:
    train_df[col] = train_df[col].fillna("")

train_df["target"] = (
    train_df["Category"].astype(str) + ":" +
    train_df["Misconception"].fillna("NA").astype(str)
)
unique_labels = sorted(train_df["target"].unique().tolist())
unique_labels_set = set(unique_labels)
fallback_labels = train_df["target"].value_counts().head(3).index.tolist()
if not fallback_labels:
    fallback_labels = ["NA:NA", "NA:NA", "NA:NA"]

# 用學生的解釋文字建立 BM25 索引
train_corpus = [tokenize_text(text) for text in train_df["StudentExplanation"]]
bm25_detector = LightweightBM25(train_corpus)
print(f"BM25 Index built successfully. Total training examples: {len(train_df)}")

# ==== 2. RAG 動態 少樣本 Prompt 建立器 ====
def build_user_prompt(question, correct_answer, student_explanation):
    return (
        "You are a math misconception classifier.\n"
        "Given the question, the correct answer, and the student's explanation, "
        "predict the final label in the format `Category:Misconception`.\n"
        "If the category is not a misconception type, use `NA` for the misconception part.\n\n"
        f"Question: {question}\n"
        f"Correct answer: {correct_answer}\n"
        f"Student explanation: {student_explanation}\n\n"
        "Return only the label."
    )

def build_dynamic_rag_prompt(row):
    # A. 現場檢索歷史相似案例 (RAG)
    q_tokens = tokenize_text(row["StudentExplanation"])
    matched_idx = bm25_detector.retrieve_top_1(q_tokens)
    matched_row = train_df.iloc[matched_idx]
    
    # B. 組裝 One-Shot 範例
    example_user = build_user_prompt(
        matched_row["QuestionText"], matched_row["MC_Answer"], matched_row["StudentExplanation"]
    )
    example_target = matched_row["target"]
    
    # C. 組裝當前測試目標
    current_user = build_user_prompt(
        row["QuestionText"], row["MC_Answer"], row["StudentExplanation"]
    )
    
    # D. 嚴格遵循 Gemma-3 聊天對齊格式拼接
    return (
        f"<start_of_turn>user\n{example_user}<end_of_turn>\n"
        f"<start_of_turn>model\n{example_target}<end_of_turn>\n"
        f"<start_of_turn>user\n{current_user}<end_of_turn>\n"
        f"<start_of_turn>model\n"
    )

# ==== Model: load + merge LoRA + eval ====
print("Loading base model in bf16 ...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"  base loaded in {time.time()-t0:.1f}s")

t0 = time.time()
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
print(f"  adapter loaded in {time.time()-t0:.1f}s")

print("Merging LoRA into base ...")
t0 = time.time()
model = model.merge_and_unload()
torch.cuda.empty_cache()  
print(f"  merge done in {time.time()-t0:.1f}s")

model.eval()
model.config.use_cache = True  
device = next(model.parameters()).device

pad_id = tokenizer.pad_token_id
if pad_id is None:
    pad_id = tokenizer.eos_token_id
    tokenizer.pad_token_id = pad_id

end_of_turn_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
stop_ids = [tokenizer.eos_token_id]
if end_of_turn_id and end_of_turn_id != tokenizer.unk_token_id:
    stop_ids.append(end_of_turn_id)

# ==== Test 讀取 ====
test_df = pd.read_csv(TEST_CSV).reset_index(drop=True)
for col in ["QuestionText", "MC_Answer", "StudentExplanation"]:
    test_df[col] = test_df[col].fillna("")
print(f"Test size: {len(test_df)}")
assert len(test_df) == len(sample)

# ==== Helpers ====
def clean_label(text):
    if not text:
        return ""
    label = text.splitlines()[0].strip()
    if " " in label:
        label = label.split(" ")[0]
    return label

@torch.no_grad()
def beam_generate_batch(prompts):
    tokenizer.padding_side = "left"
    enc = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=1200, # 稍微放寬因應 RAG 長度
    ).to(device)
    
    outputs = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        num_return_sequences=NUM_RETURN,
        do_sample=False,
        early_stopping=True,
        eos_token_id=stop_ids,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )
    prompt_len = enc["input_ids"].shape[1]
    outputs = outputs.view(len(prompts), NUM_RETURN, -1)

    results = []
    for i in range(len(prompts)):
        valid, seen = [], set()
        for k in range(NUM_RETURN):
            gen = outputs[i, k, prompt_len:]
            text = tokenizer.decode(gen, skip_special_tokens=True).strip()
            label = clean_label(text)
            if label in unique_labels_set and label not in seen:
                valid.append(label)
                seen.add(label)
        results.append(valid)
    return results

@torch.no_grad()
def score_candidates_batched(prompts, candidates_per_prompt):
    """極速原地優化版 Phase 2：全程不將巨大詞表的 logits 搬離 GPU"""
    flat_sequences = []
    flat_meta = []

    for pi, (prompt, cands) in enumerate(zip(prompts, candidates_per_prompt)):
        if not cands:
            continue
        prompt_ids = tokenizer.encode(prompt, add_special_tokens=True)
        for ci, cand in enumerate(cands):
            cand_ids = tokenizer.encode(cand, add_special_tokens=False)
            if end_of_turn_id:
                cand_ids.append(end_of_turn_id)
            else:
                cand_ids.append(tokenizer.eos_token_id)
                
            flat_sequences.append(prompt_ids + cand_ids)
            flat_meta.append((pi, ci, len(cand_ids)))

    if not flat_sequences:
        return [[] for _ in prompts]

    B = len(flat_sequences)
    max_len = max(len(s) for s in flat_sequences)

    # 安全穩定的 Right-padding
    input_ids = torch.full((B, max_len), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((B, max_len), dtype=torch.long)
    label_starts = []
    
    for j, seq in enumerate(flat_sequences):
        input_ids[j, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        attention_mask[j, :len(seq)] = 1
        label_starts.append(len(seq) - flat_meta[j][2])

    scores_per_prompt = [[0.0] * len(c) for c in candidates_per_prompt]

    # 分批 forward，原地切片處理
    for m in range(0, B, SUB_BATCH_SIZE):
        sub_input_ids = input_ids[m:m+SUB_BATCH_SIZE].to(device)
        sub_attention_mask = attention_mask[m:m+SUB_BATCH_SIZE].to(device)
        
        sub_logits = model(input_ids=sub_input_ids, attention_mask=sub_attention_mask).logits
        
        sub_meta = flat_meta[m:m+SUB_BATCH_SIZE]
        sub_label_starts = label_starts[m:m+SUB_BATCH_SIZE]
        sub_flat_sequences = flat_sequences[m:m+SUB_BATCH_SIZE]

        for idx, (pi, ci, L) in enumerate(sub_meta):
            ls = sub_label_starts[idx]
            slice_logits = sub_logits[idx, ls - 1:ls - 1 + L, :].float()
            log_probs = torch.log_softmax(slice_logits, dim=-1)
            
            target_ids = sub_flat_sequences[idx][-L:]
            target = torch.tensor(target_ids, device=device)
            
            tok_lp = log_probs.gather(1, target.unsqueeze(1)).squeeze(1)
            scores_per_prompt[pi][ci] = tok_lp.mean().item()  # 僅搬移一個常數 float 
            
    return scores_per_prompt

# ==== 主推論迴圈 ====
print("\nStart RAG-augmented inference ...")
pred_dict = {}
phase1_time = 0.0
phase2_time = 0.0
n_empty_after_filter = 0
total_cands = 0

try:
    for start in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Batch"):
        batch_df = test_df.iloc[start:start + BATCH_SIZE]
        
        # 核心改動：每一筆測試資料動態檢索 BM25 並生成自適應 RAG 提示詞
        prompts = [build_dynamic_rag_prompt(r) for _, r in batch_df.iterrows()]

        # Phase 1: Beam Search 生成候選
        t0 = time.time()
        candidates = beam_generate_batch(prompts)
        phase1_time += time.time() - t0

        # Phase 2: Log-likelihood 嚴格重排
        t0 = time.time()
        scores = score_candidates_batched(prompts, candidates)
        phase2_time += time.time() - t0

        # 挑選 Top-3 與防禦填充
        for i, (_, row) in enumerate(batch_df.iterrows()):
            cands = candidates[i]
            total_cands += len(cands)
            
            if not cands:
                n_empty_after_filter += 1
                top3 = list(fallback_labels[:3])
            else:
                ranked = sorted(zip(cands, scores[i]), key=lambda x: -x[1])
                top3 = [c for c, _ in ranked]
                for fb in fallback_labels:
                    if len(top3) >= 3:
                        break
                    if fb not in top3:
                        top3.append(fb)
            
            while len(top3) < 3:
                top3.append(fallback_labels[0])
                
            pred_dict[row["row_id"]] = " ".join(top3[:3])

except Exception as e:
    print(f"\n[CRITICAL ERROR] Inference broken: {str(e)}")
    print("Generating safe dummy submission...")
    for _, row in test_df.iterrows():
        if row["row_id"] not in pred_dict:
            pred_dict[row["row_id"]] = " ".join(fallback_labels[:3])

# ==== Diagnostic ====
print(f"\nTiming:")
print(f"  Phase 1 (beam):    {phase1_time:.1f}s")
print(f"  Phase 2 (re-rank): {phase2_time:.1f}s")
print(f"  Avg candidates per row: {total_cands/len(test_df):.2f}")
print(f"  Rows with 0 valid candidates: {n_empty_after_filter} "
      f"({n_empty_after_filter/len(test_df)*100:.1f}%)")

# ==== 產出 Submission ====
submission = sample.copy()
submission[PRED_COL] = submission[ROW_ID_COL].map(pred_dict)

if submission[PRED_COL].isna().any():
    submission[PRED_COL] = submission[PRED_COL].fillna(" ".join(fallback_labels[:3]))

print("\nValidation:")
print(f"Shape: {submission.shape}")
print(f"Any NaN: {submission.isna().any().any()}")

assert submission.shape == sample.shape
assert not submission.isna().any().any()
assert (submission[PRED_COL] != "").all()
assert (submission[PRED_COL].str.split().str.len() == 3).all()

submission.to_csv(OUTPUT_CSV, index=False)
print(f"\n[OK] Saved {OUTPUT_CSV}")

Sample shape: (3, 2)
Loading train set & building BM25 index...
BM25 Index built successfully. Total training examples: 36696
Loading base model in bf16 ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

  base loaded in 35.2s


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


  adapter loaded in 2.8s
Merging LoRA into base ...
  merge done in 0.2s
Test size: 3

Start RAG-augmented inference ...


Batch: 100%|██████████| 1/1 [00:07<00:00,  7.70s/it]


Timing:
  Phase 1 (beam):    4.6s
  Phase 2 (re-rank): 2.9s
  Avg candidates per row: 4.00
  Rows with 0 valid candidates: 0 (0.0%)

Validation:
Shape: (3, 2)
Any NaN: False

[OK] Saved /kaggle/working/submission.csv
